In [1]:
import pandas as pd
import unicodedata

In [2]:
def normalize_name_col(s):
    return (
        s.astype(str)
        .str.upper()
        .str.strip()
        .str.normalize("NFKD")
        .str.encode("ascii", errors="ignore")
        .str.decode("utf-8")
        .str.replace(r"\s+", " ", regex=True)
    )

In [3]:

### Ultimate Dataset:
ufc_master = pd.read_csv(r'C:\Users\edude\OneDrive\Área de Trabalho\Códigos\Faculdade\TCC\LabA\base_dados\processed_bronze\2.ultimate_ufc_dataset\ufc_master.csv')
### Datalabs

merged_stats_n_scorecards = pd.read_csv(r'C:\Users\edude\OneDrive\Área de Trabalho\Códigos\Faculdade\TCC\LabA\base_dados\processed_bronze\3.ufc_datalab\merged_stats_n_scorecards.csv')


C:\Users\edude\AppData\Local\Temp\ipykernel_10880\973813801.py:5: DtypeWarning: Columns (60,61) have mixed types. Specify dtype option on import or set low_memory=False.
  merged_stats_n_scorecards = pd.read_csv(r'C:\Users\edude\OneDrive\Área de Trabalho\Códigos\Faculdade\TCC\LabA\base_dados\processed_bronze\3.ufc_datalab\merged_stats_n_scorecards.csv')


In [4]:
df_base = merged_stats_n_scorecards.copy()
df_odds = ufc_master.copy()

df_base["red_key"] = normalize_name_col(df_base["red_fighter_name"])
df_base["blue_key"] = normalize_name_col(df_base["blue_fighter_name"])
df_base["date_key"] = pd.to_datetime(df_base["event_date"], errors="coerce").dt.strftime("%Y-%m-%d")

df_odds["red_key"] = normalize_name_col(df_odds["red_fighter"])
df_odds["blue_key"] = normalize_name_col(df_odds["blue_fighter"])
df_odds["date_key"] = pd.to_datetime(df_odds["date"], errors="coerce").dt.strftime("%Y-%m-%d")

In [ ]:
odds_cols = [
    "red_odds",
    "blue_odds",
    "location",
    "country",
    "total_fight_time_secs",
    "red_dec_odds",
    "blue_dec_odds",
    "red_sub_odds",
    "blue_sub_odds",
    "red_ko_odds",
    "blue_ko_odds",
    "better_rank",
    "empty_arena",
    "blue_match_weightclass_rank",
    "red_match_weightclass_rank",
    "red_women_s_flyweight_rank",
    "red_women_s_featherweight_rank",
    "red_women_s_strawweight_rank",
    "red_women_s_bantamweight_rank",
    "red_heavyweight_rank",
    "red_light_heavyweight_rank",
    "red_middleweight_rank",
    "red_welterweight_rank",
    "red_lightweight_rank",
    "red_featherweight_rank",
    "red_bantamweight_rank",
    "red_flyweight_rank",
    "red_pound_for_pound_rank",
    "blue_women_s_flyweight_rank",
    "blue_women_s_featherweight_rank",
    "blue_women_s_strawweight_rank",
    "blue_women_s_bantamweight_rank",
    "blue_heavyweight_rank",
    "blue_light_heavyweight_rank",
    "blue_middleweight_rank",
    "blue_welterweight_rank",
    "blue_lightweight_rank",
    "blue_featherweight_rank",
    "blue_bantamweight_rank",
    "blue_flyweight_rank",
    "blue_pound_for_pound_rank",
]

In [7]:
odds_cols = [col for col in odds_cols if col in df_odds.columns]

In [8]:
df_odds_merge = df_odds[
    ["red_key", "blue_key", "date_key"] + odds_cols
].drop_duplicates(
    subset=["red_key", "blue_key", "date_key"]
)

In [9]:
merged_stats_n_scorecards_enriched = df_base.merge(
    df_odds_merge,
    on=["red_key", "blue_key", "date_key"],
    how="left",
    validate="many_to_one",
    indicator="odds_match"
)

In [15]:
merged_stats_n_scorecards_enriched = pd.DataFrame(merged_stats_n_scorecards_enriched)

In [17]:
merged_stats_n_scorecards_enriched.to_csv(r'C:\Users\edude\OneDrive\Área de Trabalho\Códigos\Faculdade\TCC\LabA\base_dados\processed_silver\dim_fight.csv',sep=',',index=False)